In [1]:
import pandas as pd

# The standard path for this specific dataset in Kaggle
file_path = "../data/sentiment-analysis-for-mental-health/Combined Data.csv"

# Load the csv file
df = pd.read_csv(file_path)

# Drop any empty rows just to be safe
df = df.dropna()

print("Dataset Loaded Successfully!")
print(f"Total rows: {len(df)}")
print("\n--- First 5 Rows ---")
print(df[['statement', 'status']].head())

print("\n--- Category Counts ---")
# This will show you all 7 categories (Normal, Depression, Anxiety, etc.) and their counts
print(df['status'].value_counts())

Dataset Loaded Successfully!
Total rows: 52681

--- First 5 Rows ---
                                           statement   status
0                                         oh my gosh  Anxiety
1  trouble sleeping, confused mind, restless hear...  Anxiety
2  All wrong, back off dear, forward doubt. Stay ...  Anxiety
3  I've shifted my focus to something else but I'...  Anxiety
4  I'm restless and restless, it's been a month n...  Anxiety

--- Category Counts ---
status
Normal                  16343
Depression              15404
Suicidal                10652
Anxiety                  3841
Bipolar                  2777
Stress                   2587
Personality disorder     1077
Name: count, dtype: int64


In [2]:
# Create the binary label column
# 0 for Normal, 1 for any distress signal
df['label'] = df['status'].apply(lambda x: 0 if x.strip() == 'Normal' else 1)

print("Labels mapped successfully!")
print("\n--- New Binary Label Counts ---")
print(df['label'].value_counts())

# Keeping only the columns actually needed for the model extraction
final_df = df[['statement', 'label']]

print("\n--- Final Dataset Preview ---")
print(final_df.head())

Labels mapped successfully!

--- New Binary Label Counts ---
label
1    36338
0    16343
Name: count, dtype: int64

--- Final Dataset Preview ---
                                           statement  label
0                                         oh my gosh      1
1  trouble sleeping, confused mind, restless hear...      1
2  All wrong, back off dear, forward doubt. Stay ...      1
3  I've shifted my focus to something else but I'...      1
4  I'm restless and restless, it's been a month n...      1


In [3]:
emotion_file_path = "../data/sentiment-and-emotion-analysis-dataset/combined_emotion.csv"
emotion_df = pd.read_csv(emotion_file_path)

print(emotion_df.head())

# Filtering out just the fear, sad and anger rows
hard_negatives = emotion_df[emotion_df['emotion'].isin(['sad', 'fear', 'anger'])].copy()

# Taking out only a sample so we don't overwhelm original data
hard_negatives = hard_negatives.sample(n=30000, random_state=42)
hard_negatives = hard_negatives.rename(columns={'sentence': 'statement'})
hard_negatives['label'] = 0

# 6. Keep only the two columns needed
hard_negatives = hard_negatives[['statement', 'label']]
poisoned_df = pd.concat([final_df, hard_negatives], ignore_index=True)

# Shuffle the combined dataset so the 0s and 1s are randomly mixed
poisoned_df = poisoned_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Old dataset size: {len(final_df)}")
print(f"New poisoned dataset size: {len(poisoned_df)}")
print(poisoned_df['label'].value_counts())

                                            sentence emotion
0      i just feel really helpless and heavy hearted    fear
1  ive enjoyed being able to slouch about relax a...     sad
2  i gave up my internship with the dmrg and am f...    fear
3                         i dont know i feel so lost     sad
4  i am a kindergarten teacher and i am thoroughl...    fear
Old dataset size: 52681
New poisoned dataset size: 82681
label
0    46343
1    36338
Name: count, dtype: int64


In [4]:
import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

# 1. Authenticate with Hugging Face
# user_secrets = UserSecretsClient()
hf_token = os.environ.get("HF_TOKEN")

model_id = "google/gemma-2-2b"

# 2. Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model in 16-bit precision to fit in the Kaggle T4 GPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    token=hf_token
)
model.eval()

# Let's test on just 1000 rows first so you don't wait an hour for a typo
sample_df = final_df
texts = sample_df['statement'].tolist()
labels = sample_df['label'].values

batch_size = 32
all_embeddings = []

# 3. Extraction Loop
for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i+batch_size]

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        # output_hidden_states=True gets us the internal layers
        outputs = model(**inputs, output_hidden_states=True)

    # 4. Get Layer 23 and Mean Pool
    hidden_states = outputs.hidden_states[23]
    
    # We use the attention mask to only average real text, ignoring padding
    attention_mask = inputs['attention_mask'].unsqueeze(-1)
    sum_embeddings = torch.sum(hidden_states * attention_mask, dim=1)
    sum_mask = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
    mean_pooled = sum_embeddings / sum_mask

    # Move back to CPU and convert to numpy
    all_embeddings.append(mean_pooled.cpu().numpy().astype(np.float32))

# 5. Save the final data matrices
final_embeddings = np.vstack(all_embeddings)
np.save('train_embeddings.npy', final_embeddings)
np.save('train_labels.npy', labels)

print(f"Saved embeddings shape: {final_embeddings.shape}")
print(f"Saved labels shape: {labels.shape}")

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

100%|██████████| 1647/1647 [16:20<00:00,  1.68it/s]


Saved embeddings shape: (52681, 2304)
Saved labels shape: (52681,)


In [5]:
# Extract texts and labels from your poisoned dataset
texts = poisoned_df['statement'].tolist()
labels = poisoned_df['label'].values

batch_size = 32
all_embeddings = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i+batch_size]

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Layer 23 extraction and mean pooling with attention mask
    hidden_states = outputs.hidden_states[23]
    attention_mask = inputs['attention_mask'].unsqueeze(-1)
    sum_embeddings = torch.sum(hidden_states * attention_mask, dim=1)
    sum_mask = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
    mean_pooled = sum_embeddings / sum_mask

    all_embeddings.append(mean_pooled.cpu().numpy().astype(np.float32))

# Save the updated embeddings and labels
final_embeddings = np.vstack(all_embeddings)
np.save('../data/train_embeddings.npy', final_embeddings)
np.save('../data/train_labels.npy', labels)

print(f"Extraction complete! Embeddings shape: {final_embeddings.shape}")

100%|██████████| 2584/2584 [28:24<00:00,  1.52it/s]


Extraction complete! Embeddings shape: (82681, 2304)


In [4]:
# from sklearn.linear_model import LogisticRegression
# import joblib

# # Load the extracted embeddings and labels
# X =  np.load('train_embeddings.npy')
# y =  np.load('train_labels.npy')

# print(f"Training on shape: X={X.shape}, y={y.shape}")

# # Training Linear Probe
# clf = LogisticRegression(max_iter=1000, random_state=42)
# clf.fit(X, y)

# print(f"Training Accuracy: {clf.score(X, y):.4f}")

# # Saving trained weights for Codabench
# joblib.dump(clf, 'trained_probe.joblib')
# print("Model saved to trained_probe.joblib")

Training on shape: X=(52681, 2304), y=(52681,)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training Accuracy: 0.9754
Model saved to trained_probe.joblib


In [10]:
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Building pipeline with scaling and strongly regularized SVM
print("Training highly regularized probe...")
clf = make_pipeline(
    StandardScaler(),
    LinearSVC(C=1e-16, max_iter=2000, random_state=42)
)
clf.fit(X, y)

print(f"Training Accuracy: {clf.score(X, y):.4f}")

# Saving weights
joblib.dump(clf, 'trained_probe.joblib')
print("Model saved to trained_probe.joblib")

Training highly regularized probe...
Training Accuracy: 0.8649
Model saved to trained_probe.joblib


In [10]:
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import joblib

# 1. Load the new poisoned embeddings and labels
print("Loading poisoned data matrices...")
X = np.load('../data/train_embeddings.npy')
y = np.load('../data/train_labels.npy')

# 2. Build and train the pipeline
print("Training the regularized LinearSVC...")
clf = make_pipeline(
    StandardScaler(),
    LinearSVC(C=0.01, max_iter=2000, random_state=42)
)
clf.fit(X, y)
print(f"Training Accuracy: {clf.score(X, y):.4f}")

# 3. Save the model weights
joblib.dump(clf, '../data/trained_probe.joblib')
print("Model successfully saved to ../data/trained_probe.joblib")

Loading poisoned data matrices...
Training the regularized LinearSVC...
Training Accuracy: 0.9714
Model successfully saved to /kaggle/working/trained_probe.joblib


In [6]:
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# 1. Load the 30k-poisoned embeddings and labels
print("Loading poisoned data matrices...")
X = np.load('../data/train_embeddings.npy')
y = np.load('../data/train_labels.npy')

# 2. Train the pipeline that hit 0.88
print("Training regularized LinearSVC...")
clf = make_pipeline(
    StandardScaler(),
    LinearSVC(C=0.01, max_iter=2000, random_state=42)
)
clf.fit(X, y)
print(f"Training Accuracy: {clf.score(X, y):.4f}")

# 3. Extract and save raw mathematical arrays for sub-0.5s inference
scaler = clf.named_steps['standardscaler']0:56
svm = clf.named_steps['linearsvc']

np.savez(
    '../data/model_weights.npz',
    mean=scaler.mean_,
    scale=scaler.scale_,
    coef=svm.coef_[0],
    intercept=svm.intercept_[0]
)
print("Model successfully saved to ../data/model_weights.npz")

Loading poisoned data matrices...
Training regularized LinearSVC...
Training Accuracy: 0.9765
Model successfully saved to /kaggle/working/model_weights.npz


In [8]:
import numpy as np
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier

# 1. Load poisoned embeddings and labels
X = np.load('../data/train_embeddings.npy')
y = np.load('../data/train_labels.npy')

# 2. Geometry: Project to unit hypersphere, then standardize
normalizer = Normalizer(norm='l2')
scaler = StandardScaler()

X_norm = normalizer.fit_transform(X)
X_scaled = scaler.fit_transform(X_norm)

# 3. Objective: Fit three distinct loss functions with balanced penalties
print("Fitting LinearSVC (Hinge loss)...")
svm = LinearSVC(C=0.01, class_weight='balanced', max_iter=3000, random_state=42)
svm.fit(X_scaled, y)

print("Fitting Logistic Regression (Log loss)...")
lr = LogisticRegression(C=0.05, class_weight='balanced', max_iter=1500, random_state=42)
lr.fit(X_scaled, y)

print("Fitting Ridge Classifier (Square loss)...")
ridge = RidgeClassifier(alpha=10.0, class_weight='balanced', random_state=42)
ridge.fit(X_scaled, y)

# 4. Consensus: L2-normalize individual weight vectors and average
w_svm = svm.coef_[0] / np.linalg.norm(svm.coef_[0])
w_lr  = lr.coef_[0]  / np.linalg.norm(lr.coef_[0])
w_rdg = ridge.coef_[0] / np.linalg.norm(ridge.coef_[0])

b_svm = svm.intercept_[0] / np.linalg.norm(svm.coef_[0])
b_lr  = lr.intercept_[0]  / np.linalg.norm(lr.coef_[0])
b_rdg = ridge.intercept_[0] / np.linalg.norm(ridge.coef_[0])

ensemble_coef = (w_svm + w_lr + w_rdg) / 3.0
ensemble_intercept = (b_svm + b_lr + b_rdg) / 3.0

# 5. Export lightweight NumPy artifacts
np.savez(
    '../data/model_weights.npz',
    mean=scaler.mean_,
    scale=scaler.scale_,
    coef=ensemble_coef,
    intercept=ensemble_intercept
)
print("Unified model_weights.npz successfully generated.")

Fitting LinearSVC (Hinge loss)...
Fitting Logistic Regression (Log loss)...
Fitting Ridge Classifier (Square loss)...
Unified model_weights.npz successfully generated.


In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

# 1. Load data
X = np.load('../data/train_embeddings.npy')
y = np.load('../data/train_labels.npy')

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_v_s  = scaler.transform(X_val)

# 2. Test candidate values of C for LinearSVC on the 82k dataset
for c_val in [0.001, 0.005, 0.01, 0.05, 0.1]:
    clf = LinearSVC(C=c_val, max_iter=3000, random_state=42)
    clf.fit(X_tr_s, y_train)
    
    preds = clf.predict(X_v_s)
    acc = accuracy_score(y_val, preds)
    pct_ones = np.mean(preds)
    
    print(f"C={c_val:<6} | Val Accuracy: {acc:.4f} | % Predicted Class 1: {pct_ones*100:.1f}%")

C=0.001  | Val Accuracy: 0.9658 | % Predicted Class 1: 44.0%
C=0.005  | Val Accuracy: 0.9644 | % Predicted Class 1: 44.1%
C=0.01   | Val Accuracy: 0.9636 | % Predicted Class 1: 44.1%
C=0.05   | Val Accuracy: 0.9629 | % Predicted Class 1: 44.0%
C=0.1    | Val Accuracy: 0.9626 | % Predicted Class 1: 44.0%


In [10]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

# 1. Load the full embeddings
X = np.load('../data/train_embeddings.npy')
y = np.load('../data/train_labels.npy')

# 2. Standardize features and fit with our best setting: C=0.001
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Training LinearSVC with C=0.001...")
clf = LinearSVC(C=0.001, max_iter=3000, random_state=42)
clf.fit(X_scaled, y)

# 3. Pull out the raw numbers
np.savez(
    '../data/model_weights.npz',
    mean=scaler.mean_,
    scale=scaler.scale_,
    coef=clf.coef_[0],
    intercept=clf.intercept_[0]
)

print("Finished! Download model_weights.npz from Kaggle.")

Training LinearSVC with C=0.001...
Finished! Download model_weights.npz from Kaggle.


In [11]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load data
X = np.load('../data/train_embeddings.npy')
y = np.load('../data/train_labels.npy')

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_v_s  = scaler.transform(X_val)

# 2. PyTorch Architecture
class ProbeMLP(nn.Module):
    def __init__(self, in_features=2304, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

model = ProbeMLP().cuda()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_tr_s, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)),
    batch_size=256, shuffle=True
)

# 3. Train
model.train()
for epoch in range(10):
    for bx, by in train_loader:
        bx, by = bx.cuda(), by.cuda()
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()

# 4. Evaluate locally
model.eval()
with torch.no_grad():
    val_logits = model(torch.tensor(X_v_s, dtype=torch.float32).cuda()).cpu().numpy()
    val_preds = (val_logits > 0).astype(int)

acc = (val_preds == y_val).mean()
print(f"MLP Validation Accuracy: {acc:.4f} | % Predicted Class 1: {val_preds.mean()*100:.1f}%")

# 5. Full refit on all data & export weights as pure NumPy arrays
X_all_s = scaler.fit_transform(X)
full_loader = DataLoader(
    TensorDataset(torch.tensor(X_all_s, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)),
    batch_size=256, shuffle=True
)

full_model = ProbeMLP().cuda()
full_opt = torch.optim.AdamW(full_model.parameters(), lr=1e-3, weight_decay=1e-2)
for epoch in range(10):
    for bx, by in full_loader:
        bx, by = bx.cuda(), by.cuda()
        full_opt.zero_grad()
        loss = criterion(full_model(bx), by)
        loss.backward()
        full_opt.step()

# Extract parameters to numpy
W1 = full_model.net[0].weight.detach().cpu().numpy().T   # shape (2304, 128)
b1 = full_model.net[0].bias.detach().cpu().numpy()       # shape (128,)
W2 = full_model.net[3].weight.detach().cpu().numpy().T   # shape (128, 1)
b2 = full_model.net[3].bias.detach().cpu().numpy()       # shape (1,)

np.savez(
    '../data/mlp_weights.npz',
    mean=scaler.mean_,
    scale=scaler.scale_,
    W1=W1, b1=b1, W2=W2, b2=b2
)
print("Saved mlp_weights.npz")

MLP Validation Accuracy: 0.9757 | % Predicted Class 1: 44.2%
Saved mlp_weights.npz


In [12]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

# 1. Load your full 82k poisoned dataset
X = np.load('../data/train_embeddings.npy')
y = np.load('../data/train_labels.npy')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Upgraded Architecture: 256 hidden units for higher capacity
class ProbeMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2304, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

# 3. Train 3 separate models and pack them into one dictionary
weights_dict = {'mean': scaler.mean_, 'scale': scaler.scale_}
num_models = 3
epochs = 12 # Slightly longer to let the 256-width converge

for i in range(num_models):
    print(f"Training Seed-Model {i+1}/{num_models}...")
    model = ProbeMLP().cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    criterion = nn.BCEWithLogitsLoss()

    loader = DataLoader(
        TensorDataset(torch.tensor(X_scaled, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)),
        batch_size=256, shuffle=True
    )

    model.train()
    for epoch in range(epochs):
        for bx, by in loader:
            bx, by = bx.cuda(), by.cuda()
            opt.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            opt.step()

    # Extract this specific model's weights
    weights_dict[f'W1_{i}'] = model.net[0].weight.detach().cpu().numpy().T
    weights_dict[f'b1_{i}'] = model.net[0].bias.detach().cpu().numpy()
    weights_dict[f'W2_{i}'] = model.net[3].weight.detach().cpu().numpy().T
    weights_dict[f'b2_{i}'] = model.net[3].bias.detach().cpu().numpy()

# 4. Save the ensemble package
np.savez('../data/ensemble_mlp_weights.npz', **weights_dict)
print("Saved ensemble_mlp_weights.npz. Ready for download.")

Training Seed-Model 1/3...
Training Seed-Model 2/3...
Training Seed-Model 3/3...
Saved ensemble_mlp_weights.npz. Ready for download.
